In [ ]:
# Milestone 4 - Features extraction testing
from src.board import Board
from src.features import FeatureExtractor

board = Board(4, 3)
board.grid = [
    ["X", "X", None, None],
    [None, "O", None, None],
    [None, None, "O", None],
    [None, None, None, None],
]

fx = FeatureExtractor()
features = fx.extract(board, my_player="X", opponent_player="O")

for k in sorted(features.keys()):
    print(f"{k}: {features[k]}")

blocked_windows: 2
empty_cells: 12
k: 3
my_center_control: 0
my_immediate_wins: 1
my_marks: 2
my_open_1: 3
my_open_2: 1
my_open_lines: 4
my_two_way_threats: 1
my_winning_windows: 0
n: 4
neutral_windows: 8
opp_center_control: 2
opp_immediate_wins: 1
opp_marks: 2
opp_open_1: 9
opp_open_2: 1
opp_open_lines: 10
opp_two_way_threats: 6
opp_winning_windows: 0
total_windows: 24


In [2]:
# Milestone 5 - Data collection test
from src.data_collection import collect_dataset

rows = collect_dataset(
    num_games=20,
    n=4,
    k=3,
    matchup="tactical_vs_random",
    seed=42,
)

print("rows:", len(rows))
print("sample keys:", sorted(rows[0].keys())[:15], "...")
print("sample row:", {k: rows[0][k] for k in ['game_id', 'turn_index', 'current_player', 'final_winner', 'final_outcome']})

rows: 112
sample keys: ['blocked_windows', 'board_before_move', 'current_player', 'empty_cells', 'final_outcome', 'final_winner', 'game_id', 'k', 'move_col', 'move_row', 'my_center_control', 'my_immediate_wins', 'my_marks', 'my_open_1', 'my_open_2'] ...
sample row: {'game_id': 0, 'turn_index': 0, 'current_player': 'X', 'final_winner': 'X', 'final_outcome': 1}


In [1]:
# DATA ANALYSIS SCRIPT
# Load milestone 5's CSV, group by final_outcome, and print average values for each feature
import csv
from collections import defaultdict

# Path to your dataset
csv_path = "report/results/m5_dataset.csv"

# Features to analyze (add/remove as needed)
feature_keys = [
    "my_immediate_wins", "opp_immediate_wins",
    "my_open_2", "opp_open_2",  # for k=3; use my_open_{k-1} for your k
    "my_two_way_threats", "opp_two_way_threats",
    "my_center_control", "opp_center_control"
]

# Load data
rows = []
with open(csv_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Convert all features to int
        for k in feature_keys + ["final_outcome"]:
            if k in row:
                row[k] = int(row[k])
        rows.append(row)

# Group by outcome
groups = defaultdict(list)
for row in rows:
    groups[row["final_outcome"]].append(row)

# Print averages
for outcome, group in sorted(groups.items()):
    print(f"\nOutcome {outcome} (n={len(group)}):")
    for k in feature_keys:
        avg = sum(r[k] for r in group) / len(group)
        print(f"  {k:20s}: {avg:7.3f}")


Outcome -1 (n=765):
  my_immediate_wins   :   0.014
  opp_immediate_wins  :   0.703
  my_open_2           :   0.014
  opp_open_2          :   0.703
  my_two_way_threats  :   0.769
  opp_two_way_threats :   2.532
  my_center_control   :   0.150
  opp_center_control  :   1.165

Outcome 1 (n=918):
  my_immediate_wins   :   0.533
  opp_immediate_wins  :   0.248
  my_open_2           :   0.533
  opp_open_2          :   0.248
  my_two_way_threats  :   1.731
  opp_two_way_threats :   1.821
  my_center_control   :   0.971
  opp_center_control  :   0.256


## Milestone 6 — Heuristic Evaluation Function

**Goal:** Build an interpretable, weighted-sum heuristic that scores any board state for the AI. The weights are grounded in the M5 dataset — features that appear more often in winning positions get positive weight; features that appear more often in losing positions get negative weight.

**Approach:**
1. Load the M5 dataset and compute per-feature averages grouped by outcome.
2. Pick the features with the largest win/loss difference as the basis for weights.
3. Implement `HeuristicEvaluator` in `src/heuristics.py` with hand-tuned weights derived from those averages.
4. Integrate the heuristic into Alpha-Beta search via a depth cutoff (`get_best_move_heuristic` in `src/ai.py`).

In [6]:
# Milestone 6 — Step 1: Feature correlation table
# Shows which features most reliably separate wins from losses,
# and how the heuristic weights were chosen.
import csv
from collections import defaultdict

csv_path = "report/results/m5_dataset.csv"
feature_keys = [
    "my_immediate_wins",  "opp_immediate_wins",
    "my_open_2",          "opp_open_2",
    "my_two_way_threats", "opp_two_way_threats",
    "my_center_control",  "opp_center_control",
]

rows = []
with open(csv_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        for k in feature_keys + ["final_outcome"]:
            if k in row:
                row[k] = int(row[k])
        rows.append(row)

groups = defaultdict(list)
for row in rows:
    groups[row["final_outcome"]].append(row)

wins   = groups[1]
losses = groups[-1]

print(f"Dataset: {len(wins)} win rows, {len(losses)} loss rows\n")
print(f"{'Feature':<24} {'Win avg':>9} {'Loss avg':>9} {'Diff (W-L)':>11}  Weight in heuristic")
print("-" * 80)

weights = {
    "my_immediate_wins":   2.0,
    "opp_immediate_wins": -2.0,
    "my_open_2":           1.0,
    "opp_open_2":         -1.0,
    "my_two_way_threats":  0.5,
    "opp_two_way_threats":-1.0,
    "my_center_control":   1.0,
    "opp_center_control": -1.0,
}

for feat in feature_keys:
    w_avg = sum(r[feat] for r in wins)   / len(wins)
    l_avg = sum(r[feat] for r in losses) / len(losses)
    diff  = w_avg - l_avg
    w     = weights[feat]
    print(f"{feat:<24} {w_avg:>9.3f} {l_avg:>9.3f} {diff:>+11.3f}    {w:>+.1f}")

print()
print("Interpretation:")
print("  Diff > 0 → feature is higher in wins   → positive weight")
print("  Diff < 0 → feature is higher in losses → negative weight")
print("  Weight magnitude is proportional to the size of the difference.")

Dataset: 918 win rows, 765 loss rows

Feature                    Win avg  Loss avg  Diff (W-L)  Weight in heuristic
--------------------------------------------------------------------------------
my_immediate_wins            0.533     0.014      +0.518    +2.0
opp_immediate_wins           0.248     0.703      -0.455    -2.0
my_open_2                    0.533     0.014      +0.518    +1.0
opp_open_2                   0.248     0.703      -0.455    -1.0
my_two_way_threats           1.731     0.769      +0.962    +0.5
opp_two_way_threats          1.821     2.532      -0.711    -1.0
my_center_control            0.971     0.150      +0.820    +1.0
opp_center_control           0.256     1.165      -0.909    -1.0

Interpretation:
  Diff > 0 → feature is higher in wins   → positive weight
  Diff < 0 → feature is higher in losses → negative weight
  Weight magnitude is proportional to the size of the difference.


In [7]:
# Milestone 6 — Step 2: Heuristic scores on crafted boards
# Verifies that the heuristic assigns sensible signs and magnitudes
# to positions a human would intuitively recognise as good, bad, or neutral.
from src.board import Board
from src.heuristics import HeuristicEvaluator

def make_board(grid, n, k):
    b = Board(n, k)
    b.grid = [row[:] for row in grid]
    return b

h = HeuristicEvaluator()

examples = [
    ("X has won (full row)",           [["X","X","X"],[None,"O",None],[None,None,"O"]]),
    ("O has won (full column)",        [["X","O",None],["X","O",None],[None,"O","X"]]),
    ("Draw (full board, no winner)",   [["X","O","X"],["O","X","O"],["O","X","O"]]),
    ("X has immediate win threat",     [["X","X",None],["O",None,None],[None,None,None]]),
    ("O has immediate win threat",     [["O","O",None],["X",None,None],[None,None,None]]),
    ("Early game, X controls center",  [[None,None,None],[None,"X",None],[None,None,None]]),
]

print(f"{'Board description':<38} {'Score':>8}  Interpretation")
print("-" * 72)
for desc, grid in examples:
    score = h.evaluate(make_board(grid, 3, 3), "X", "O")
    interp = "X winning" if score > 100 else "O winning" if score < -100 else "X slight edge" if score > 0 else "O slight edge" if score < 0 else "neutral"
    print(f"{desc:<38} {score:>8.1f}  {interp}")

print("\nPositive score = good for X (maximizer).  Negative = good for O (minimizer).")
print("±1000 = terminal win/loss.  Small values = positional advantage.")

Board description                         Score  Interpretation
------------------------------------------------------------------------
X has won (full row)                      999.0  X winning
O has won (full column)                  -998.0  O winning
Draw (full board, no winner)                1.0  X slight edge
X has immediate win threat                  3.5  X slight edge
O has immediate win threat                 -4.0  O slight edge
Early game, X controls center               1.0  X slight edge

Positive score = good for X (maximizer).  Negative = good for O (minimizer).
±1000 = terminal win/loss.  Small values = positional advantage.


In [8]:
# Milestone 6 — Step 3: Heuristic-backed AI vs exhaustive AI
# Compares get_best_move_heuristic (depth-limited) against get_best_move_ab
# (exhaustive) on two scenarios. Both should agree on the correct move while
# the heuristic version visits far fewer nodes.
from src.game import Game
from src.ai import MinimaxAI

ai = MinimaxAI()

scenarios = [
    {
        "desc": "X to move — immediate win available at (0,2)",
        "grid": [["X","X",None],["O",None,None],[None,None,None]],
        "expect": (0, 2),
    },
    {
        "desc": "X to move — must block O's immediate win at (1,2)",
        "grid": [["X",None,None],["O","O",None],[None,None,None]],
        "expect": (1, 2),
    },
]

for s in scenarios:
    game = Game(n=3, k=3, player1="X", player2="O")
    game.board.grid = [row[:] for row in s["grid"]]

    move_h  = ai.get_best_move_heuristic(game, "X", "O", max_depth=4)
    nodes_h = ai.nodes_explored_h

    move_ab  = ai.get_best_move_ab(game, "X", "O")
    nodes_ab = ai.nodes_explored_ab

    agree = "✓ agree" if move_h == move_ab else "✗ DIFFER"
    print(f"Scenario: {s['desc']}")
    print(f"  Heuristic AB  → move {move_h}  ({nodes_h:,} nodes)")
    print(f"  Exhaustive AB → move {move_ab}  ({nodes_ab:,} nodes)   {agree}")
    print()

Scenario: X to move — immediate win available at (0,2)
  Heuristic AB  → move (0, 2)  (94 nodes)
  Exhaustive AB → move (0, 2)  (161 nodes)   ✓ agree

Scenario: X to move — must block O's immediate win at (1,2)
  Heuristic AB  → move (1, 2)  (190 nodes)
  Exhaustive AB → move (1, 2)  (310 nodes)   ✓ agree

